This file extracts the text from the manual scoring pdf's, converts it into an array and excel and then compares it to the YASA scorings.

In [3]:
# prerequisites + extract text from manual scoring pdf's

#%pip install pdfplumber
from pathlib import Path
import re
import numpy as np
import pandas as pd
import pdfplumbaer

FOLDER = Path("Pilotdaten/Scoring_Files (do not delete)") # folder with pilot data (scored)

pdf_files = sorted(FOLDER.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF file(s)")

for pdf_path in pdf_files:
    with pdfplumber.open(pdf_path) as pdf:
        text = "\n\n".join(page.extract_text() or "" for page in pdf.pages)
    print(f"===== {pdf_path.name} =====")
    print(text[:100])   # nur den Anfang anzeigen

Found 46 PDF file(s)
===== 04-49-15-4B-D3-6E_Axo-Device-075_20250707_204037_000009_RvL.pdf =====
UNIVERSITÄT BERN
Bolligenstrasse 111
3000 Bern 60
Polyman Report Generator by Remus von Lucadou
Prin
===== 04-49-15-4B-D3-6E_Axo-Device-075_20250708_205434_000010_RvL.pdf =====
UNIVERSITÄT BERN
Bolligenstrasse 111
3000 Bern 60
Polyman Report Generator by Remus von Lucadou
Prin
===== 11-E2-7B-2A-A0-71_Axo-Device-071_20250421_204304_000019_RvL.pdf =====
UNIVERSITÄT BERN
Bolligenstrasse 111
3000 Bern 60
Polyman Report Generator by Remus von Lucadou
Prin
===== 11-E2-7B-2A-A0-71_Axo-Device-071_20250504_210352_000029_RvL.pdf =====
UNIVERSITÄT BERN
Bolligenstrasse 111
3000 Bern 60
Polyman Report Generator by Remus von Lucadou
Prin
===== 11-E2-7B-2A-A0-71_Axo-Device-071_20250521_213303_000042_RvL.pdf =====
UNIVERSITÄT BERN
Bolligenstrasse 111
3000 Bern 60
Polyman Report Generator by Remus von Lucadou
Prin
===== 11-E2-7B-2A-A0-71_Axo-Device-071_20250610_210154_000046_RvL.pdf =====
UNIVERSITÄT BERN
B

In [8]:
# convert text into array/excel

FOLDER = Path("Pilotdaten/Scoring_Files (do not delete)")

SHIFT = r"(?:\s+(\d+)\s+epochs shift to W)?"

# (regex, [(column name, kind), ...])
# kinds: "text" | "hms" (3 groups -> minutes) | "hm" (2 groups -> minutes) | "num" (1 group)
PATTERNS = [
    (r"EDF file\(s\)\s+(\S+)",                        [("file_name", "text")]),
    (r"TST:\s*(\d+)h(\d+)m(\d+)s",                    [("TST_min", "hms")]),
    (r"Sleep efficiency:\s*(\d+)%",                   [("SleepEff_pct", "num")]),
    (r"Sleep latency:\s*(\d+)h(\d+)m",                [("SleepLat_min", "hm")]),
    (r"REM latency:\s*(\d+)h(\d+)m",                  [("REMLat_min", "hm")]),
    (r"WASO:\s*(\d+)h(\d+)m",                         [("WASO_min", "hm")]),
    (r"\bW\s+(\d+)h(\d+)m(\d+)s",                     [("W_min", "hms")]),
    (r"\bN1\s+(\d+)h(\d+)m(\d+)s\s+(\d+)%",          [("N1_min", "hms"), ("N1_pct", "num")]),
    (r"\bN2\s+(\d+)h(\d+)m(\d+)s\s+(\d+)%" + SHIFT,  [("N2_min", "hms"), ("N2_pct", "num"), ("N2_shiftW", "num")]),
    (r"\bN3\s+(\d+)h(\d+)m(\d+)s\s+(\d+)%" + SHIFT,  [("N3_min", "hms"), ("N3_pct", "num"), ("N3_shiftW", "num")]),
    (r"\bREM\s+(\d+)h(\d+)m(\d+)s\s+(\d+)%" + SHIFT, [("REM_min", "hms"), ("REM_pct", "num"), ("REM_shiftW", "num")]),
    (r"Number of arousals:\s*(\d+)",                  [("n_arousals", "num")]),
    (r"Arousal index \(TST\):\s*(\d+(?:[.,]\d+)?)",   [("arousal_index", "num")]),
]
COLUMNS = [c for _, specs in PATTERNS for c, _ in specs]


def num(s):
    return np.nan if s is None else float(s.replace(",", "."))


def extract_row(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        text = "\n".join(page.extract_text() or "" for page in pdf.pages)

    row = {}
    for pattern, specs in PATTERNS:
        m = re.search(pattern, text)
        g = 1  # index of the next regex group to consume
        for col, kind in specs:
            if kind == "text":
                row[col] = m.group(g) if m else pdf_path.name
                g += 1
            elif kind == "hms":
                row[col] = (int(m.group(g)) * 60 + int(m.group(g + 1))
                            + int(m.group(g + 2)) / 60) if m else np.nan
                g += 3
            elif kind == "hm":
                row[col] = (int(m.group(g)) * 60 + int(m.group(g + 1))) if m else np.nan
                g += 2
            else:  # "num"
                row[col] = num(m.group(g)) if m else np.nan
                g += 1
    return row, text


pdf_files = sorted(FOLDER.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF file(s)")

rows = []
for pdf_path in pdf_files:
    try:
        row, text = extract_row(pdf_path)
        missing = [c for c in COLUMNS if isinstance(row[c], float) and np.isnan(row[c])
                   and not c.endswith("shiftW")]   # shiftW may legitimately be absent
        if missing:
            print(f"WARNING {pdf_path.name}: not found -> {missing}")
        rows.append(row)
    except Exception as e:
        print(f"FAILED {pdf_path.name}: {e}")

df = pd.DataFrame(rows, columns=COLUMNS)
data = df.to_numpy()

print(df)
print(data.shape)

df.to_excel("sleep_summary.xlsx", index=False)

Found 46 PDF file(s)
WARNING 28-4E-38-D8-04-0F_Axo-Device-058_20240411_201854_000215_3_RvL.pdf: not found -> ['arousal_index']
WARNING 28-4E-38-D8-04-0F_Axo-Device-058_20240415_202320_000216_2_RvL.pdf: not found -> ['arousal_index']
WARNING 28-4E-38-D8-04-0F_Axo-Device-058_20240416_204916_000217_3_RvL.pdf: not found -> ['arousal_index']
WARNING 28-4E-38-D8-04-0F_Axo-Device-058_20241112_002155_000248_RvL.pdf: not found -> ['arousal_index']
WARNING 28-4E-38-D8-04-0F_Axo-Device-058_20241114_235745_000251_RvL.pdf: not found -> ['arousal_index']
WARNING 2A-B6-A2-D4-B6-EB_Sleep-Loop-37_20240220_220207_000127_RvL.pdf: not found -> ['arousal_index']
                                            file_name  TST_min  SleepEff_pct  \
0   04-49-15-4B-D3-6E_Axo-Device-075_20250707_2040...    329.0          64.0   
1   04-49-15-4B-D3-6E_Axo-Device-075_20250708_2054...    367.5          74.0   
2   11-E2-7B-2A-A0-71_Axo-Device-071_20250421_2043...    415.5          83.0   
3   11-E2-7B-2A-A0-71_Axo-Devi

To fit YASA file to manual file: variable names comparison

| Name Manual Scooring | Name YASA Scoring Acc |
|----------------------|-----------------------|
| TST_min | TST |
| SleepEff_pct | SE |
| SleepLat_min | SOL |
| REMLat_min | Lat_REM |
| WASO_min | WASO |
| W_min |  |
| N1_min | N1 |
| N2_min | N2 |
| N3_min | N3 |
| REM_min | REM |
|  | NREM |
| N1_pct | %N1 |
| N2_pct | %N2 |
| N3_pct  | %N3 |
| REM_pct | %REM |
|  | %NREM |
| N2_shift |  |
| N3_shift |  |
| REM_shift |  |
| n_arousals |  |
| arousal_index |  |
|  | TIB |
|  | SPT |
|  | Lat_N1 |
|  | Lat_N2 |
|  | Lat_N3 |
|  | SME |

YASA Output and what it corresponds to in the manual scorings:

- Time in Bed (TIB): total duration of the hypnogram.  -> lights off - lights on
- Sleep Period Time (SPT): duration from first to last period of sleep. -> not present in manual scoring
- Wake After Sleep Onset (WASO): duration of wake periods within SPT. -> WASO
- Total Sleep Time (TST): total sleep duration in SPT. -> TST
- Sleep Onset Latency (SOL): Latency to first epoch of any sleep.  -> SleepLat
- SOL 5min: Latency to 5 minutes of persistent sleep (any stage).  -> not present in manual scoring
- REM latency: latency to first REM sleep.  -> REM latency
- Sleep Efficiency (SE): TST / TIB * 100 (%).  -> SleepEff
- Sleep Maintenance Efficiency (SME): TST / SPT * 100 (%).  -> not present in manual scoring
- Sleep Fragmentation Index: number of transitions from sleep to wake / hours of TST  -> not present in manual scoring
- Sleep stages amount and proportion of TST  -> for W, N1, N2, N3, REM: total and % of TST

In [19]:
# comparison YASA to manual scorings

YASA_FOLDER = Path("Pilotdaten/YASA_scored")

# manual name -> YASA name (variables without a counterpart are left out)
MAPPING = {
    "TST_min":      "TST",
    "SleepEff_pct": "SE",
    "SleepLat_min": "SOL",
    "REMLat_min":   "Lat_REM",
    "WASO_min":     "WASO",
    "N1_min":       "N1",
    "N2_min":       "N2",
    "N3_min":       "N3",
    "REM_min":      "REM",
    "N1_pct":       "%N1",
    "N2_pct":       "%N2",
    "N3_pct":       "%N3",
    "REM_pct":      "%REM",
}

# ---------- load manual scorings ----------
manual = pd.read_excel('sleep_summary_manualscorings.xlsx')

def clean_id(name):
    """Normalise a file name so manual and YASA files can be matched."""
    s = Path(str(name)).stem.replace("*", "")
    return s.lower()

manual["_key"] = manual["file_name"].map(clean_id)

# ---------- load YASA files ----------
def read_any(path):
    if path.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(path)
    return pd.read_csv(path, sep=None, engine="python")   # auto-detects , or ;

yasa_files = sorted(p for p in YASA_FOLDER.iterdir()
                    if p.suffix.lower() in (".xlsx", ".xls"))
print(f"Found {len(yasa_files)} YASA file(s)")

rows = []
for path in yasa_files:
    try:
        yasa = read_any(path)
        if len(yasa) > 1:
            print(f"NOTE {path.name}: {len(yasa)} rows, using the first one")
        y = yasa.iloc[0]

        # find the matching manual row (manual ID contained in YASA file name or vice versa)
        ykey = clean_id(path.name)
        match = manual[manual["_key"].map(lambda k: k in ykey or ykey in k)]
        if match.empty:
            print(f"WARNING {path.name}: no matching manual scoring found")
            continue
        if len(match) > 1:
            print(f"WARNING {path.name}: {len(match)} manual matches, using the first")
        m = match.iloc[0]

        row = {"participant": m["file_name"], "yasa_file": path.name}
        for man_col, yasa_col in MAPPING.items():
            mv = pd.to_numeric(m.get(man_col, np.nan), errors="coerce")
            yv = pd.to_numeric(y.get(yasa_col, np.nan), errors="coerce")
            # only where both have data (and manual is not 0, to avoid dividing by zero)
            if pd.notna(mv) and pd.notna(yv) and mv != 0:
                row[man_col.replace("_min", "").replace("_pct", "_pct")] = yv / mv
            else:
                row[man_col.replace("_min", "").replace("_pct", "_pct")] = np.nan
        rows.append(row)
    except Exception as e:
        print(f"FAILED {path.name}: {e}")

comparison = pd.DataFrame(rows)
accuracy_array = comparison.drop(columns=["participant", "yasa_file"]).to_numpy(dtype=float)

# check all comparison values
display(comparison)
print(accuracy_array.shape)

# summary row per variable (mean over participants, ignoring NaN)
print("\nMean comparison per variable:")
display(comparison.drop(columns=["participant", "yasa_file"]).mean())

comparison.to_excel("comparison.xlsx", index=False)

Found 21 YASA file(s)
FAILED 1C-AA-17-D5-58-87_Axo-Device-070_20250313_224755_000042_metrics.xlsx: [Errno 13] Permission denied: 'Pilotdaten\\YASA_scored\\1C-AA-17-D5-58-87_Axo-Device-070_20250313_224755_000042_metrics.xlsx'


,participant,yasa_file,TST,SleepEff_pct,SleepLat,REMLat,WASO,N1,N2,N3,REM,N1_pct,N2_pct,N3_pct,REM_pct
0,04-49-15-4B-D3-6E_Axo-Device-075_20250708_2054...,04-49-15-4B-D3-6E_Axo-Device-075_20250708_2054...,1.027211,1.033708,1.000000,1.267606,0.758621,2.882353,0.847100,5.100000,0.714286,2.781457,0.829105,6.754967,0.709555
1,11-E2-7B-2A-A0-71_Axo-Device-071_20250421_2043...,11-E2-7B-2A-A0-71_Axo-Device-071_20250421_2043...,0.978339,0.982465,1.018519,1.285714,1.145455,0.621053,0.964000,1.195652,1.125000,0.659734,0.988110,1.230012,1.172129
2,11-E2-7B-2A-A0-71_Axo-Device-071_20250504_2103...,11-E2-7B-2A-A0-71_Axo-Device-071_20250504_2103...,0.988270,0.985380,1.043860,1.938596,1.017544,0.448980,1.080952,1.314286,1.007752,0.466299,1.086436,1.364985,1.015149
3,11-E2-7B-2A-A0-71_Axo-Device-071_20250521_2133...,11-E2-7B-2A-A0-71_Axo-Device-071_20250521_2133...,0.911905,0.907561,1.031250,1.202532,1.077381,0.619048,0.929825,1.179487,0.916667,0.652742,1.020845,1.334494,1.013669
4,11-E2-7B-2A-A0-71_Axo-Device-071_20250610_2101...,11-E2-7B-2A-A0-71_Axo-Device-071_20250610_2101...,0.969565,0.976357,1.106061,2.397959,0.806250,0.558140,0.955399,1.634921,0.965217,0.597907,0.981243,1.710679,0.975996
5,11-E2-7B-2A-A0-71_Axo-Device-071_20250622_2136...,11-E2-7B-2A-A0-71_Axo-Device-071_20250622_2136...,0.966867,0.961726,1.022727,1.723077,0.416058,0.768421,1.033058,1.152778,0.828358,0.812194,1.062022,1.175304,0.864486
6,11-E2-7B-2A-A0-71_Axo-Device-071_20250623_2119...,11-E2-7B-2A-A0-71_Axo-Device-071_20250623_2119...,0.988732,0.990406,1.022222,NaN,1.058140,0.639175,0.782516,0.951389,NaN,0.630851,0.792109,0.975783,NaN
7,11-E2-7B-2A-A0-71_Axo-Device-071_20250630_2149...,11-E2-7B-2A-A0-71_Axo-Device-071_20250630_2149...,0.970861,0.967810,0.892857,1.217742,0.639785,1.146341,0.918367,1.300000,0.823944,1.165819,0.952627,1.330150,0.840095
8,1C-AA-17-D5-58-87_Axo-Device-070_20250226_2114...,1C-AA-17-D5-58-87_Axo-Device-070_20250226_2114...,1.012048,1.006109,0.953704,1.632530,0.841667,0.744186,1.008000,0.720430,2.133333,0.793651,1.004464,0.712160,2.116402
9,1C-AA-17-D5-58-87_Axo-Device-070_20250414_2149...,1C-AA-17-D5-58-87_Axo-Device-070_20250414_2149...,1.000000,0.996715,1.055556,1.185000,0.968750,1.192308,1.040404,0.854406,1.070175,1.174242,1.040404,0.844697,1.094498


(20, 13)

Mean comparison per variable:


TST             0.998871
SleepEff_pct    0.998716
SleepLat        1.023915
REMLat          1.391995
WASO            0.888683
N1              0.951059
N2              0.955813
N3              1.534253
REM             1.248786
N1_pct          0.955732
N2_pct          0.958661
N3_pct          1.591464
REM_pct         1.239363
dtype: float64